# Bloc 3 : Analyse Prédictive Non-Supervisée (Uber Pickups)

## Étape 1 : Préparation des Données et Pipeline (ETL)
**Objectif :** Charger l'historique des trajets, extraire les composantes temporelles (Feature Engineering) et isoler un périmètre de test (Start Small). 
Afin de garantir la validité des calculs de distances géométriques pour nos futurs algorithmes, les coordonnées GPS subiront une standardisation stricte.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
# Ces 2 lignes doivent venir AVANT scikit-learn (elles évitent un plantage de K-Means sur Mac)
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [3]:
# 1. Ingestion de la donnée (Stratégie Start Small : Avril 2014)
print("Chargement des données d'Avril en cours...")
df_apr = pd.read_csv('../data/uber-raw-data-apr14.csv')

# 2. Feature Engineering Temporel
df_apr['Date/Time'] = pd.to_datetime(df_apr['Date/Time'])
df_apr['DayOfWeek'] = df_apr['Date/Time'].dt.day_name()
df_apr['Hour'] = df_apr['Date/Time'].dt.hour

# 3. Échantillonnage : Isolation du pic de demande (Jeudi à 18h)
df_jeudi_18h = df_apr[(df_apr['DayOfWeek'] == 'Thursday') & (df_apr['Hour'] == 18)].copy()

# 4. Pipeline Scikit-Learn : Isolation et Normalisation des coordonnées spatiales
coords = df_jeudi_18h[['Lat', 'Lon']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

# 5. Vérification
print(f"✅ Pipeline terminé. Nombre de trajets isolés pour l'entraînement : {len(coords_scaled)}")

Chargement des données d'Avril en cours...
✅ Pipeline terminé. Nombre de trajets isolés pour l'entraînement : 6637


## Étape 2 : La Preuve Mathématique (K-Means & Méthode du Coude)
**Objectif :** Déterminer mathématiquement le nombre optimal de "Hot-Zones" (K) à recommander aux chauffeurs Uber pour notre créneau de test (Jeudi 18h). 
L'algorithme K-Means exige que l'on définisse le nombre de zones à l'avance. Pour éviter un choix arbitraire, nous mesurons l'Inertie (la distance géographique intra-cluster) pour chaque K, de 2 à 10.

In [4]:
import os
import plotly.express as px
from sklearn.cluster import KMeans

#On force KMeans à n'utiliser qu'un seul processeur pour éviter le crash OpenMP
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 1. Initialisation de la mémoire
inerties = []
valeurs_k = range(2, 11)

print("Entraînement de l'algorithme K-Means en cours (Test de K=2 à K=10)...")

# 2. La boucle d'entraînement itérative
for k in valeurs_k:
    # Instanciation du modèle avec 'k' zones
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    # Entraînement sur la carte GPS normalisée (issue de l'étape 1)
    kmeans.fit(coords_scaled)
    # Sauvegarde de l'erreur géographique
    inerties.append(kmeans.inertia_)
    print(f"K={k} : inertie = {kmeans.inertia_:.2f}")

# 3. Restitution visuelle (Le Graphique)
fig_coude = px.line(
    x=list(valeurs_k), 
    y=inerties, 
    markers=True, 
    title="Optimisation Spatiale : Méthode du Coude (Elbow Method)",
    labels={"x": "Nombre de Hot-Zones recommandées (K)", "y": "Inertie (Dispersion spatiale)"}
)

fig_coude.update_layout(template="plotly_white")
fig_coude.show()

Entraînement de l'algorithme K-Means en cours (Test de K=2 à K=10)...
K=2 : inertie = 9297.14
K=3 : inertie = 6363.99
K=4 : inertie = 4707.04
K=5 : inertie = 3756.69
K=6 : inertie = 3095.66
K=7 : inertie = 2522.47
K=8 : inertie = 2098.69
K=9 : inertie = 1752.05
K=10 : inertie = 1477.84


## Étape 3 : Déploiement Géospatial (K-Means avec K=5)
**Objectif :** Appliquer la conclusion de la méthode du coude (K=5) pour segmenter les clients en 5 "Hot-Zones". 
Nous allons projeter ces clusters mathématiques sur une carte interactive de New York via `Plotly Mapbox`, afin de fournir un outil visuel d'aide à la décision pour le dispatching des chauffeurs.

In [5]:
import plotly.express as px
import pandas as pd

print("🗺️ Génération de la cartographie des 5 Hot-Zones...")

# 1. Instanciation du modèle définitif avec le K optimal (5 zones)
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)

# 2. Entraînement et attribution immédiate d'une étiquette (0 à 4) à chaque trajet
# On crée une nouvelle colonne 'Cluster' dans notre tableau d'origine
df_jeudi_18h['Cluster'] = kmeans_final.fit_predict(coords_scaled)

# 3. Création de la carte interactive avec Plotly
fig_map = px.scatter_map(
    df_jeudi_18h, 
    lat="Lat", 
    lon="Lon", 
    color=df_jeudi_18h['Cluster'].astype(str), # Force le cluster en texte pour avoir des couleurs distinctes
    map_style="carto-positron", # Style de fond de carte clair et professionnel
    zoom=9, 
    title="K-Means (comparaison) : 5 zones, toutes les courses incluses (jeudi 18 h)",
    color_discrete_sequence=px.colors.qualitative.Bold # Palette de couleurs contrastée
)

# 4. Affichage du rendu final
fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_map.show()

🗺️ Génération de la cartographie des 5 Hot-Zones...


## Étape 4 : Modélisation par Densité (DBSCAN) et Filtrage du Bruit
**Objectif :** L'algorithme K-Means a démontré ses limites en forçant l'inclusion des valeurs aberrantes (courses isolées), créant des zones trop vastes pour un dispatching efficace.
Nous mettons en concurrence ce modèle avec **DBSCAN**, un algorithme basé sur la densité. Son avantage métier est double : il n'a pas besoin de connaître le nombre de zones à l'avance, et il exclut automatiquement les courses isolées (classées comme "Bruit" ou Outliers) pour ne cibler que les véritables nids de demande.

In [6]:
from sklearn.cluster import DBSCAN

print("🔍 Analyse de densité DBSCAN en cours...")

# 1. Instanciation de DBSCAN
# eps=0.2 : La distance maximale entre deux clients pour être dans la même zone
# min_samples=15 : Il faut au moins 15 clients proches pour créer une "Hot-Zone" officielle
dbscan = DBSCAN(eps=0.2, min_samples=15)

# 2. Entraînement et prédiction
# DBSCAN assignera le cluster "-1" à toutes les courses isolées (le bruit)
df_jeudi_18h['Cluster_DBSCAN'] = dbscan.fit_predict(coords_scaled)

# 3. Création de la carte du modèle retenu
fig_dbscan = px.scatter_map(
    df_jeudi_18h, 
    lat="Lat", 
    lon="Lon", 
    color=df_jeudi_18h['Cluster_DBSCAN'].astype(str),
    map_style="carto-positron",
    zoom=9, 
    title="DBSCAN (modèle retenu) : 8 zones, les courses isolées sont le groupe -1 (jeudi 18 h)",
    color_discrete_sequence=px.colors.qualitative.Vivid
)

fig_dbscan.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_dbscan.show()

🔍 Analyse de densité DBSCAN en cours...


## Étape 5 : Stratégie Business & Extraction des Coordonnées GPS
**Objectif :** Transformer notre modèle mathématique DBSCAN en produit exploitable par l'application Uber. 
Nous allons exclure le bruit (Cluster -1) et calculer le point central exact de chaque Hot-Zone valide. Ces coordonnées (Latitude, Longitude) constituent les points de rassemblement que le Dispatcher enverra aux chauffeurs pour garantir une attente client inférieure à 5 minutes.

In [7]:
print("Extraction des coordonnées GPS des Hot-Zones (Modèle DBSCAN)...")

# 1. On exclut le bruit (Les points roses, Cluster -1)
df_zones_valides = df_jeudi_18h[df_jeudi_18h['Cluster_DBSCAN'] != -1]

# 2. On calcule la position moyenne (Centroïde) pour chaque vraie Hot-Zone
centroides_dbscan = df_zones_valides.groupby('Cluster_DBSCAN')[['Lat', 'Lon']].mean().reset_index()

# 3. Formatage de la réponse pour la direction des opérations
print("\nCOORDONNÉES À INJECTER DANS L'APPLICATION CHAUFFEUR :")
display(centroides_dbscan.round(4))

Extraction des coordonnées GPS des Hot-Zones (Modèle DBSCAN)...

COORDONNÉES À INJECTER DANS L'APPLICATION CHAUFFEUR :


,Cluster_DBSCAN,Lat,Lon
0,0,40.7186,-73.9561
1,1,40.7464,-73.9867
2,2,40.7729,-73.8858
3,3,40.7707,-73.8664
4,4,40.6450,-73.7857
5,5,40.6929,-74.1787
6,6,40.6877,-73.9838
7,7,40.7217,-74.0367


## Étape 6 : Généralisation "Grow Big" (Analyse Hebdomadaire)
**Objectif :** Le cahier des charges exige de "décrire les zones à risque pour chaque jour de la semaine". 
Après avoir validé la pertinence de notre modèle DBSCAN sur un échantillon (Start Small : Jeudi 18h), nous analysons la volumétrie globale du mois d'Avril pour identifier les jours nécessitant le déploiement massif de cet algorithme.

In [8]:
print("Analyse de la demande hebdomadaire (Grow Big)...")

# 1. Agrégation : On compte le nombre de trajets par jour de la semaine
jours_ordre = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_jours = df_apr['DayOfWeek'].value_counts().reindex(jours_ordre).reset_index()
df_jours.columns = ['Jour_de_la_Semaine', 'Nombre_de_Trajets']

# 2. Création du graphique Plotly
fig_jours = px.bar(
    df_jours, 
    x='Jour_de_la_Semaine', 
    y='Nombre_de_Trajets',
    title="Volume de commandes Uber par jour (Avril 2014)",
    color='Nombre_de_Trajets',
    color_continuous_scale='Reds'
)

fig_jours.update_layout(template="plotly_white")
fig_jours.show()
print(df_jours)

Analyse de la demande hebdomadaire (Grow Big)...


  Jour_de_la_Semaine  Nombre_de_Trajets
0             Monday              60861
1            Tuesday              91185
2          Wednesday             108631
3           Thursday              85067
4             Friday              90303
5           Saturday              77218
6             Sunday              51251


## Étape 7 : Les zones chaudes pour chaque jour de la semaine (à 18h)

**Objectif :** Le cahier des charges demande de décrire les zones à risque pour chaque jour. Je refais exactement le même calcul que pour le jeudi (mêmes réglages : `eps=0.2` et `min_samples=15`), mais pour les 7 jours, toujours à 18h.

**Comment lire le résultat :** pour chaque jour, on voit le nombre de courses, le nombre de zones chaudes trouvées, et le nombre de courses isolées (le « bruit »). Plus il y a de zones, plus la demande est répartie à plusieurs endroits.

In [9]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

jours_a_analyser = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
resultats_par_jour = {}

for jour in jours_a_analyser:
    # Même logique de filtrage que pour le jeudi, à l'heure de pointe (18h)
    df_jour = df_apr[(df_apr['DayOfWeek'] == jour) & (df_apr['Hour'] == 18)].copy()

    coords_jour = df_jour[['Lat', 'Lon']]
    scaler = StandardScaler()
    coords_jour_scaled = scaler.fit_transform(coords_jour)

    dbscan = DBSCAN(eps=0.2, min_samples=15)
    df_jour['Cluster'] = dbscan.fit_predict(coords_jour_scaled)

    nb_zones = df_jour['Cluster'].nunique() - (1 if -1 in df_jour['Cluster'].values else 0)
    nb_bruit = (df_jour['Cluster'] == -1).sum()

    resultats_par_jour[jour] = {"trajets": len(df_jour), "zones": nb_zones, "bruit": nb_bruit}
    print(f"{jour} 18h : {len(df_jour)} trajets → {nb_zones} zones chaudes, {nb_bruit} points isolés")

Monday 18h : 4725 trajets → 7 zones chaudes, 256 points isolés
Tuesday 18h : 7089 trajets → 7 zones chaudes, 299 points isolés
Wednesday 18h : 8334 trajets → 10 zones chaudes, 325 points isolés
Thursday 18h : 6637 trajets → 8 zones chaudes, 266 points isolés
Friday 18h : 7258 trajets → 7 zones chaudes, 368 points isolés
Saturday 18h : 6165 trajets → 6 zones chaudes, 300 points isolés
Sunday 18h : 2795 trajets → 6 zones chaudes, 170 points isolés


## Étape 8 : Validation Objective du Choix de K (Score de Silhouette comparatif)

**Objectif :** La Méthode du Coude reste une heuristique visuelle. Pour trancher
mathématiquement entre les valeurs de K proches de notre choix (K=5), nous
calculons le Score de Silhouette pour plusieurs valeurs de K et comparons
objectivement leur qualité de séparation.

In [10]:
from sklearn.metrics import silhouette_score

print("Comparaison du Score de Silhouette pour plusieurs valeurs de K...")
for k in [4, 5, 6, 7]:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_test.fit_predict(coords_scaled)
    score = silhouette_score(coords_scaled, labels)
    print(f"K={k} : Score de Silhouette = {score:.3f}")

Comparaison du Score de Silhouette pour plusieurs valeurs de K...
K=4 : Score de Silhouette = 0.528
K=5 : Score de Silhouette = 0.533
K=6 : Score de Silhouette = 0.521
K=7 : Score de Silhouette = 0.524


## Étape 9 : Test de Sensibilité des Hyperparamètres DBSCAN

**Objectif :** Prouver que le choix de eps=0.2 et min_samples=15 n'est pas
arbitraire, en observant comment le nombre de zones et de points de bruit
évolue si l'on fait varier chaque paramètre autour de sa valeur retenue.

In [11]:
print("--- Sensibilité sur eps (min_samples fixé à 15) ---")
for eps_test in [0.1, 0.15, 0.2, 0.25, 0.3]:
    dbscan_test = DBSCAN(eps=eps_test, min_samples=15)
    labels_test = dbscan_test.fit_predict(coords_scaled)
    nb_zones = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    nb_bruit = list(labels_test).count(-1)
    print(f"eps={eps_test} → {nb_zones} zones, {nb_bruit} points de bruit")

print("\n--- Sensibilité sur min_samples (eps fixé à 0.2) ---")
for min_samples_test in [10, 15, 20, 25]:
    dbscan_test = DBSCAN(eps=0.2, min_samples=min_samples_test)
    labels_test = dbscan_test.fit_predict(coords_scaled)
    nb_zones = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    nb_bruit = list(labels_test).count(-1)
    print(f"min_samples={min_samples_test} → {nb_zones} zones, {nb_bruit} points de bruit")

--- Sensibilité sur eps (min_samples fixé à 15) ---
eps=0.1 → 11 zones, 529 points de bruit
eps=0.15 → 9 zones, 358 points de bruit
eps=0.2 → 8 zones, 266 points de bruit
eps=0.25 → 6 zones, 227 points de bruit
eps=0.3 → 5 zones, 182 points de bruit

--- Sensibilité sur min_samples (eps fixé à 0.2) ---
min_samples=10 → 9 zones, 221 points de bruit
min_samples=15 → 8 zones, 266 points de bruit
min_samples=20 → 8 zones, 330 points de bruit
min_samples=25 → 7 zones, 359 points de bruit
